# LLAMA 3.1

In [11]:
from collections import Counter
import json

def cargar_datos_entrenamiento(ruta_archivo):
    with open(ruta_archivo, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    X = []
    y = []
    ids = []

    for key, value in data.items():
        tweet_text = value['tweet']
        votos = value['labels_task1']
        
        conteo = Counter(votos)
        label_mayoritaria, frecuencia = conteo.most_common(1)[0]
        total_votos = len(votos)
        if frecuencia <= total_votos / 2:
            continue 
            
        X.append(tweet_text)
        y.append(label_mayoritaria)
        ids.append(value['id_EXIST'])

    return X, y, ids



ruta = 'dataset_task1_exist2025/training.json' 
try:
    X_train, y_train, ids_train = cargar_datos_entrenamiento(ruta)
    
    print(f"Procesados {len(X_train)} tweets correctamente.")
    print(f"Ejemplo - Texto: {X_train[0][:50]}...")
    print(f"Ejemplo - Label: {y_train[0]}")

except FileNotFoundError:
    print("Error: No se encontró el archivo training.json. Revisa la ruta.")

Procesados 6064 tweets correctamente.
Ejemplo - Texto: @TheChiflis Ignora al otro, es un capullo.El probl...
Ejemplo - Label: YES


In [ ]:
import torch
from transformers import pipeline, BitsAndBytesConfig
from huggingface_hub import login
from tqdm import tqdm
from sklearn.metrics import classification_report

login(token="login_token")

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

pipe = pipeline(
    "text-generation",
    model=model_id,
    device_map="auto",
    model_kwargs={
        "quantization_config": bnb_config,
        "torch_dtype": torch.bfloat16
    }
)

def obtener_respuesta_llama(prompt):
    outputs = pipe(
        prompt,
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=pipe.tokenizer.eos_token_id,
        return_full_text=False
    )

    respuesta = outputs[0]["generated_text"].strip().upper()

    if respuesta.startswith("YES"):
        return "YES"
    elif respuesta.startswith("NO"):
        return "NO"
    else:
        return "NO"

# Zero-shot
print("Iniciando Zero-shot...")

y_pred_zero = []
muestreo_test = X_train[5:55]
y_true_test = y_train[5:55]

for tweet in tqdm(muestreo_test):
    prompt = f"""Eres un clasificador bilingüe (español/inglés) experto en detectar sexismo. 
Responde solo con YES o NO. Clasifica el siguiente tweet:

Tweet: "{tweet}" ->"""
    y_pred_zero.append(obtener_respuesta_llama(prompt))

# Few-shot
print("Iniciando Few-shot...")

ejemplos_few_shot = """
Tweet: "@TheChiflis Ignora al otro, es un capullo. El problema con este youtuber denuncia el acoso... cuando no afecta a la gente de izquierdas. Por ejemplo, en su video sobre el gamergate presenta como 'normal' el acoso que reciben Fisher, Anita o Zöey cuando hubo hasta amenazas de bomba." -> YES
Tweet: "@ultimonomada_ Si comicsgate se parece en algo a gamergate pues muy bien por el acoso. Y si se está haciendo un sabotaje porque hay personajes que no os gustan entonces gracias por darme la razón. Sois unos lloricas ofendidos." -> NO
Tweet: "@Steven2897 Lee sobre Gamergate, y como eso ha cambiado la manera en la cual nos comunicamos en el internet. Los fanboys de Halo están tóxicos pero los fanboys de otras comunidades/juegos también han querido coger pauta con eso 🤷🏾‍♂️" -> NO
Tweet: "ik when mandy says “you look like a whore” i look cute as FUCK" -> YES
Tweet: "@EvansVibes_ Oh yes, Watson at times when he sleeps with me , wakes me up, so I wake him up to have him move a bit and stop Lmao https://t.co/DqlYWA4czp" -> NO
"""

y_pred_few = []

for tweet in tqdm(muestreo_test):
    prompt_few = f"""Eres un clasificador bilingüe (español/inglés) experto en detectar sexismo. 
Responde solo con YES o NO. Clasifica los tweets basándote en estos ejemplos:

{ejemplos_few_shot}

Tweet: '{tweet}' ->"""
    y_pred_few.append(obtener_respuesta_llama(prompt_few))

print("\nRESULTADOS ZERO-SHOT: ")
print(classification_report(y_true_test, y_pred_zero))

print("\nRESULTADOS FEW-SHOT:")
print(classification_report(y_true_test, y_pred_few))

Loading weights:   1%|▏         | 4/291 [00:01<03:32,  1.35it/s, Materializing param=model.layers.0.mlp.down_proj.weight]  c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 291/291 [00:13<00:00, 21.83it/s, Materializing param=model.norm.weight]                               


Iniciando Zero-shot...


  0%|          | 0/50 [00:00<?, ?it/s]Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 50/50 [04:05<00:00,  4.91s/it]


Iniciando Few-shot...


100%|██████████| 50/50 [12:13<00:00, 14.67s/it]


RESULTADOS ZERO-SHOT: 
              precision    recall  f1-score   support

          NO       0.80      1.00      0.89        40
         YES       0.00      0.00      0.00        10

    accuracy                           0.80        50
   macro avg       0.40      0.50      0.44        50
weighted avg       0.64      0.80      0.71        50


RESULTADOS FEW-SHOT:
              precision    recall  f1-score   support

          NO       0.82      0.90      0.86        40
         YES       0.33      0.20      0.25        10

    accuracy                           0.76        50
   macro avg       0.58      0.55      0.55        50
weighted avg       0.72      0.76      0.74        50




c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\franm\Desktop\SEXISM_TWEETS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitaliz